# 🧠 Implementing Transformers for Text Generation

In this notebook I build a **decoder-style Transformer** from the ground up in TensorFlow/Keras and train it to generate Shakespeare-like text, one token at a time. I handle the full pipeline myself: load and vectorise the text, window it into input/target pairs, stack Transformer blocks with positional encoding, train, and then sample new text with temperature control.

## 📋 Overview

Text generation is **autoregressive**: the model predicts the next token from everything before it, then feeds its own output back in as context. A Transformer does this by letting every position attend to the others through self-attention — no recurrence needed.

🎯 **What I do here:**

| Step | What I build |
|---|---|
| 📥 Part 1 | Load the Shakespeare corpus and vectorise it into integer tokens |
| 🔢 Part 2 | Slice the token stream into input/target sequence pairs (shifted by one) |
| 🏗️ Part 3 | Build the Transformer: embedding + positional encoding + stacked blocks |
| 🎯 Part 4 | Train with sparse categorical cross-entropy and early stopping |
| 📤 Part 5 | Generate text autoregressively with temperature sampling |
| 🧪 Part 6 | Experiments: sequence length, LR scheduler, longer generation |

**📡 Engineering analogy.** An autoregressive language model is a **predictive coder**, like the linear predictors in speech codecs (LPC / DPCM): it predicts the next sample from a window of past samples, and the better the prediction, the lower the residual entropy. Here the "samples" are word tokens and the predictor is a Transformer instead of a fixed linear filter.

## 🧩 Theory

### 🔢 Language modelling objective

The model learns a probability distribution over the next token given the previous ones. For a sequence $x_1, \dots, x_T$ it factorises the joint probability autoregressively:

$$p(x_1, \dots, x_T) = \prod_{t=1}^{T} p(x_t \mid x_1, \dots, x_{t-1})$$

Training minimises the **cross-entropy** between the predicted distribution and the true next token:

$$\mathcal{L} = -\frac{1}{T}\sum_{t=1}^{T} \log p(x_t \mid x_{<t})$$

I use `sparse_categorical_crossentropy` because targets are **integer token IDs**, not one-hot vectors — same loss, just a memory-efficient encoding.

### ➕ Positional encoding

Self-attention is order-agnostic, so I inject position with fixed sinusoids of geometrically spaced frequencies:

$$PE_{(pos,\, 2i)} = \sin\!\left(\frac{pos}{10000^{\,2i/d}}\right), \qquad PE_{(pos,\, 2i+1)} = \cos\!\left(\frac{pos}{10000^{\,2i/d}}\right)$$

📡 This is a **multi-tone timestamp** — a bank of reference frequencies that uniquely fingerprints each position, much like pilot tones marking time in a frame.

### 🎯 Temperature sampling

At generation time I divide the logits by a temperature $\tau$ before softmax:

$$p_i = \frac{\exp(z_i / \tau)}{\sum_j \exp(z_j / \tau)}$$

Low $\tau$ (<1) sharpens the distribution → safer, more repetitive text; high $\tau$ (>1) flattens it → more diverse, riskier text. 📡 It's a **noise-temperature knob** on the sampler: turn it down for a clean, deterministic signal, up to inject controlled randomness.

## Part 1 — 📥 Environment & Data

I pin TensorFlow 2.16.2 for reproducibility, then load Andrej Karpathy's classic **tiny-Shakespeare** corpus and turn it into integer tokens with a `TextVectorization` layer.

In [ ]:
%%capture
!pip install tensorflow==2.16.2
!pip install pandas
!pip install scikit-learn

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.utils import get_file 

### 📥 Load the corpus

`get_file` downloads and caches the text; I decode it to a UTF-8 string and preview the start to confirm it loaded.

In [ ]:
# Load the dataset
path_to_file = get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')

# Preview the dataset
print(text[:1000]) 

### 🔢 Vectorise the text

`TextVectorization` builds a vocabulary of the `vocab_size` most frequent tokens and maps the corpus to a 1-D array of integer IDs. 📡 Think of it as a **quantiser with a codebook**: each word is mapped to the nearest codeword (its token ID), and rare words fall outside the codebook (mapped to the out-of-vocabulary token).

| Parameter | Value | Meaning |
|---|---|---|
| `vocab_size` | 10000 | Max distinct tokens kept |
| `seq_length` | 100 | Context window the model sees |
| `output_mode` | `'int'` | Emit integer IDs (not counts/TF-IDF) |

In [ ]:
# Preprocess the dataset
vocab_size = 10000
seq_length = 100

# Adapt TextVectorization to full text
vectorizer = TextVectorization(max_tokens=vocab_size, output_mode='int')
text_ds = tf.data.Dataset.from_tensor_slices([text]).batch(1)
vectorizer.adapt(text_ds)

# Vectorize the text
vectorized_text = vectorizer([text])[0]
print("Vectorized text shape:", vectorized_text.shape)
print("First 10 vectorized tokens:", vectorized_text.numpy()[:10]) 

`adapt` scans the corpus to learn the vocabulary; the result is one long token vector representing the whole text — the raw material I'll slice into training windows next.

## Part 2 — 🔢 Input & Target Sequences

For next-token prediction, the **target is the input shifted by one position**. A sliding window of length `seq_length` over the token stream gives me $X$ (positions $i \dots i+99$) and $Y$ (positions $i+1 \dots i+100$).

In [ ]:
def create_sequences(text, seq_length):
    input_seqs = []
    target_seqs = []
    for i in range(len(text) - seq_length):
        input_seq = text[i:i + seq_length]
        target_seq = text[i + 1:i + seq_length + 1]
        input_seqs.append(input_seq)
        target_seqs.append(target_seq)
    return np.array(input_seqs), np.array(target_seqs)

# Generate sequences
X, Y = create_sequences(vectorized_text.numpy(), seq_length)

# Check if sequences are correctly generated
print("Number of sequences generated:", len(X))
print("Sample input sequence:", X[0] if len(X) > 0 else "No sequences generated")

# Check if X and Y are not empty
assert X.size > 0, "Input data X is empty"
assert Y.size > 0, "Target data Y is empty"
X = tf.convert_to_tensor(X)
Y = tf.convert_to_tensor(Y)
print("Shape of X:", X.shape)
print("Shape of Y:", Y.shape)

Each $X$ row is a 100-token context and each $Y$ row is the same window slid one step ahead. The `assert` guards catch the common silent failure where the window is longer than the text and nothing gets generated. 📡 This is exactly a **tapped delay line with a one-sample-ahead target** — the predictive-coding setup from the overview.

## Part 3 — 🏗️ Build the Transformer

I define two classes: a reusable `TransformerBlock` (self-attention + feed-forward, each with Add & Norm), and the full `TransformerModel` (embedding → positional encoding → stacked blocks → vocabulary projection).

### 🔄 Transformer block

Here I use Keras's built-in `MultiHeadAttention` rather than hand-rolling it. The block is the standard "Add & Norm" sandwich: attention sub-layer, then a position-wise feed-forward sub-layer, each wrapped in a residual connection and [[layer_normalization]], with dropout for regularisation.

### 🔢 Full model with positional encoding

The model embeds tokens into `embed_dim` vectors, **adds** the sinusoidal positional encoding, passes the result through `num_layers` Transformer blocks, and finally projects each position to a `vocab_size`-logit vector — the score for every possible next token. The `get_angles`/`positional_encoding` helpers implement the sinusoid formula from the theory section.

In [ ]:
from tensorflow.keras.layers import Embedding, MultiHeadAttention, Dense, LayerNormalization, Dropout
from tensorflow.keras.models import Model

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

class TransformerModel(Model):  # Model is now properly imported
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim, num_layers, seq_length):
        super(TransformerModel, self).__init__()
        self.embedding = Embedding(vocab_size, embed_dim)
        self.pos_encoding = self.positional_encoding(seq_length, embed_dim)
        self.transformer_blocks = [TransformerBlock(embed_dim, num_heads, ff_dim) for _ in range(num_layers)]
        self.dense = Dense(vocab_size)

    def positional_encoding(self, seq_length, embed_dim):
        angle_rads = self.get_angles(np.arange(seq_length)[:, np.newaxis], np.arange(embed_dim)[np.newaxis, :], embed_dim)
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
        pos_encoding = angle_rads[np.newaxis, ...]
        return tf.cast(pos_encoding, dtype=tf.float32)

    def get_angles(self, pos, i, embed_dim):
        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(embed_dim))
        return pos * angle_rates

    def call(self, inputs, training=False):
        seq_len = tf.shape(inputs)[1]
        x = self.embedding(inputs)
        x += self.pos_encoding[:, :seq_len, :]
        for transformer_block in self.transformer_blocks:
            x = transformer_block(x, training=training)  # Pass training argument correctly
        output = self.dense(x)
        return output

### ⚙️ Instantiate and compile

I pick a modest configuration, then force the model to build its weights by calling it once on a dummy batch (subclassed models don't know their shapes until first call). Compiling with Adam + sparse categorical cross-entropy completes the setup.

| Hyperparameter | Value |
|---|---|
| `embed_dim` | 256 |
| `num_heads` | 4 |
| `ff_dim` | 512 |
| `num_layers` | 4 |

In [ ]:
# Hyperparameters
embed_dim = 256
num_heads = 4
ff_dim = 512
num_layers = 4

# Build the Transformer model
model = TransformerModel(vocab_size, embed_dim, num_heads, ff_dim, num_layers, seq_length)

# Provide input shape to build the model by passing a dummy input with maxval specified
_ = model(tf.random.uniform((1, seq_length), maxval=vocab_size, dtype=tf.int32))

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

# Summary of the model
model.summary()

## Part 4 — 🎯 Train the Transformer

To keep runtime short I cap the data at 10,000 sequences and train for 2 epochs — enough to see the loss drop and generate plausible text, though far from convergence. `EarlyStopping` halts training if the loss stops improving and restores the best weights.

In [ ]:
!pip install matplotlib

> ℹ️ The full corpus is large, so I deliberately reduce it to 10,000 samples and 2 epochs to minimise execution time. For real quality I'd train much longer.

In [ ]:
X = X[:10000]
Y = Y[:10000]

In [ ]:
# Import necessary libraries for training visualization
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping

# Early stopping callback to stop training if the loss doesn't improve
early_stopping = EarlyStopping(monitor='loss', patience=2, restore_best_weights=True)

# Train the transformer model on the full input and target sequences
history = model.fit(X, Y, epochs=2, batch_size=32, callbacks=[early_stopping])

# Plot training loss to monitor model performance over epochs
plt.plot(history.history['loss'])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()

The loss curve should slope downward across the two epochs, confirming the model is learning the statistics of the text. 📡 Falling cross-entropy means a **lower-entropy prediction residual** — the predictor is getting better at anticipating the next token.

## Part 5 — 📤 Generate Text

Generation is a loop: vectorise the seed string, predict next-token logits, apply temperature, **sample** from the resulting distribution (rather than taking the argmax, which would be repetitive), append the token, slide the window, and repeat. I pad or truncate the seed to match `seq_length`.

In [ ]:
def generate_text(model, start_string, num_generate=100, temperature=1.0):
    # Convert the start string to a vectorized format
    input_eval = vectorizer([start_string]).numpy()

    # Ensure the input length is the same as the model's expected input shape
    if input_eval.shape[1] < seq_length:
        # Pad the input if it's shorter than the expected sequence length
        padding = np.zeros((1, seq_length - input_eval.shape[1]))
        input_eval = np.concatenate((padding, input_eval), axis=1)
    elif input_eval.shape[1] > seq_length:
        # Truncate the input if it's longer than the expected sequence length
        input_eval = input_eval[:, -seq_length:]

    input_eval = tf.convert_to_tensor(input_eval)

    # Initialize an empty list to store generated text
    text_generated = []

    # Start generating text
    for i in range(num_generate):
        # Make predictions using the model
        predictions = model(input_eval)

        # Remove only the batch dimension, keep the logits as 2D (batch_size, vocab_size)
        predictions = predictions[0]  # This should be of shape [vocab_size]

        # Apply temperature to predictions
        predictions = predictions / temperature

        # Use a categorical distribution to predict the next word
        predicted_id = tf.random.categorical(predictions, num_samples=1)[0, 0].numpy()

        # Update the input tensor to include the predicted word, maintaining the sequence length
        input_eval = np.append(input_eval.numpy(), [[predicted_id]], axis=1)  # Append predicted token
        input_eval = input_eval[:, -seq_length:]  # Keep only the last `seq_length` tokens
        input_eval = tf.convert_to_tensor(input_eval)  # Convert back to tensor

        # Append the predicted word to the generated text
        text_generated.append(vectorizer.get_vocabulary()[predicted_id])

    # Return the generated text starting from the initial seed
    return start_string + ' ' + ' '.join(text_generated)

# Generate text with temperature control
start_string = "To be, or not to be"
generated_text = generate_text(model, start_string, temperature=0.7)  # Lower temperature for more focused predictions
print(generated_text)

With only 2 epochs the output won't be coherent Shakespeare, but it should already capture word-level patterns and structure. The `temperature=0.7` setting biases toward higher-probability tokens for more focused output. 📡 `tf.random.categorical` is the **stochastic decision stage** — sampling from the posterior over tokens rather than hard-thresholding.

## Part 6 — 🧪 Experiments

Three variations to probe how design choices affect training and generation. Everything else stays fixed.

### 🧪 Experiment 1 — Shorter sequence length (100 → 50)

🎯 **Goal:** rebuild the pipeline with `seq_length = 50` and compare training loss. A shorter context means more sequences but a smaller window of history per prediction — 📡 a **shorter prediction filter**: cheaper and faster, but blind to longer-range dependencies. I re-vectorise, regenerate sequences, rebuild the model at the new length, and retrain.

In [ ]:
# Preprocess the dataset
vocab_size = 10000
seq_length = 50

# Adapt TextVectorization to full text
vectorizer = TextVectorization(max_tokens=vocab_size, output_mode='int')
text_ds = tf.data.Dataset.from_tensor_slices([text]).batch(1)
vectorizer.adapt(text_ds)

# Vectorize the text
vectorized_text = vectorizer([text])[0]
print("Vectorized text shape:", vectorized_text.shape)
print("First 10 vectorized tokens:", vectorized_text.numpy()[:10])

X, Y = create_sequences(vectorized_text.numpy(), seq_length)

# Check if sequences are correctly generated
print("Number of sequences generated:", len(X))
print("Sample input sequence:", X[0] if len(X) > 0 else "No sequences generated")

# Check if X and Y are not empty
assert X.size > 0, "Input data X is empty"
assert Y.size > 0, "Target data Y is empty"
X = tf.convert_to_tensor(X)
Y = tf.convert_to_tensor(Y)
print("Shape of X:", X.shape)
print("Shape of Y:", Y.shape)
X = X[:10000]
Y = Y[:10000]
# Hyperparameters
embed_dim = 256
num_heads = 4
ff_dim = 512
num_layers = 4

# Build the Transformer model
model = TransformerModel(vocab_size, embed_dim, num_heads, ff_dim, num_layers, seq_length)

# Provide input shape to build the model by passing a dummy input with maxval specified
_ = model(tf.random.uniform((1, seq_length), maxval=vocab_size, dtype=tf.int32))

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

# Summary of the model
model.summary()
# Early stopping callback to stop training if the loss doesn't improve
early_stopping = EarlyStopping(monitor='loss', patience=2, restore_best_weights=True)

# Train the transformer model on the full input and target sequences
history = model.fit(X, Y, epochs=2, batch_size=32, callbacks=[early_stopping])

# Plot training loss to monitor model performance over epochs
plt.plot(history.history['loss'])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()

### 🧪 Experiment 2 — Learning rate scheduler

🎯 **Goal:** add a scheduler that halves the learning rate every 10 epochs and retrain. Decaying the LR lets the optimiser take big steps early then settle into a minimum — 📡 like **coarse-then-fine tuning**: large adjustments to lock on, then small ones to refine. (At only 2 epochs the decay won't trigger, but the wiring is what matters.)

In [ ]:
# Define a learning rate scheduler
def scheduler(epoch, lr):
    if epoch % 10 == 0 and epoch != 0:
        lr = lr * 0.5
    return lr


callback = tf.keras.callbacks.LearningRateScheduler(scheduler)


# Train the model with the learning rate scheduler
history = model.fit(X, Y, epochs=2, batch_size=64, callbacks=[callback])


# Plot the training loss
plt.plot(history.history['loss'])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss with Learning Rate Scheduler')
plt.show() 

### 🧪 Experiment 3 — Generate longer text (200 tokens)

🎯 **Goal:** modify `generate_text` to produce 200 tokens. This version keeps only the last 5 tokens of context per step (`input_eval[:, -5:]`) and reshapes predictions for the categorical sampler. The longer the rollout, the more the model's small errors compound — 📡 **drift in a feedback loop**, where each prediction is fed back as input and deviations accumulate over time.

In [ ]:
def generate_text(model, start_string, num_generate=200):
    # Convert the start string to numbers (vectorize)
    input_eval = vectorizer([start_string]).numpy()

    # Ensure the input tensor has the correct shape
    input_eval = tf.convert_to_tensor(input_eval[:, -5:])  # Ensure it has a shape of (1, 5)

    text_generated = []

    for i in range(num_generate):
        # Make predictions using the model
        predictions = model(input_eval)

        # Ensure predictions is a matrix with shape [batch_size, num_classes]
        predictions = tf.squeeze(predictions, 0)  # Remove the batch dimension
        predictions = tf.expand_dims(predictions, 0)  # Add back a batch dimension for categorical

        # Use a categorical distribution to predict the next word
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy()

        # Update the input tensor to include the predicted word, maintaining the sequence length
        input_eval = np.append(input_eval.numpy(), [[predicted_id]], axis=1)  # Append predicted token
        input_eval = input_eval[:, -5:]  # Keep only the last 5 tokens to match input shape
        input_eval = tf.convert_to_tensor(input_eval)  # Convert back to tensor

        # Add the predicted word to the generated text
        text_generated.append(vectorizer.get_vocabulary()[predicted_id])

    return start_string + ' ' + ' '.join(text_generated)


# Generate longer text
start_string = "To be, or not to be"
generated_text = generate_text(model, start_string)

print(generated_text)

## 📊 Summary

| 🧩 Component | Role | 📡 Engineering analogy |
|---|---|---|
| `TextVectorization` | Text → integer token IDs | Quantiser with a codebook |
| Sliding-window sequences | Input + next-token target | Tapped delay line, 1-step-ahead target |
| Token embedding | Token ID → dense vector | Mapping codeword to feature space |
| Positional encoding | Inject order via sinusoids | Multi-tone timestamp / pilot tones |
| `MultiHeadAttention` | Each token attends to the context | Adaptive correlator bank |
| Residual + LayerNorm | Stable, normalised deep stack | Bypass line + automatic gain control |
| Dense → vocab logits | Score every possible next token | Decision statistic per codeword |
| Sparse cat. cross-entropy | Loss on integer targets | Prediction-residual entropy |
| Temperature sampling | Diversity vs. focus knob | Noise-temperature control on the sampler |

✅ **What I built and learned:**
- Built an **autoregressive Transformer** language model end-to-end in Keras.
- Used `TextVectorization`, sliding-window targets, and sinusoidal positional encoding.
- Trained with `sparse_categorical_crossentropy` + early stopping, and generated text with temperature sampling.
- Explored how sequence length, LR scheduling, and rollout length affect behaviour.

| Hyperparameter | Value used |
|---|---|
| `vocab_size` | 10000 |
| `seq_length` | 100 (50 in Exp 1) |
| `embed_dim` | 256 |
| `num_heads` | 4 |
| `ff_dim` | 512 |
| `num_layers` | 4 |
| Loss / optimiser | sparse categorical cross-entropy / Adam |

## 🧪 Sandbox

Space to push further. Ideas worth trying:

- 🎯 **Sweep temperature** (0.2, 0.7, 1.0, 1.5) on the same seed and compare coherence vs. diversity.
- 📈 **Train longer** — bump epochs to 20+ and use the full corpus; watch the loss and sample quality improve.
- ⚙️ **Add a causal mask** to `MultiHeadAttention` (`use_causal_mask=True`) so each position only attends to the past — the proper decoder-only setup, preventing the model from "seeing the future" during training.
- 🔢 **Greedy vs. sampling vs. top-k** decoding — compare argmax, categorical sampling, and top-$k$ sampling.
- 🔄 **Fix the Exp-3 context window** — it truncates to 5 tokens, far below `seq_length`; align it to the full context and see how generation changes.

In [ ]:
# 🧪 Sandbox — experiment freely here
